# Lab 4 – Generation có trích dẫn

Truy xuất: pipeline tốt nhất của Lab 3 (rewrite → lọc sản phẩm → hybrid3 → rerank 20 ứng viên → **top-5**).
Sinh: `gpt-4.1-mini`, temperature 0, **structured output** (JSON schema): `status` (`tra_loi` | `khong_du_thong_tin`), `answer` có trích dẫn `[C#]`, `citations` (id + trích nguyên văn).

Hậu kiểm tự động:
- trích dẫn có tồn tại không (id hợp lệ), câu trả lời có trích dẫn không
- trích nguyên văn (`quote`) có thật sự nằm trong đoạn được trích không
- đoạn được trích có phải đoạn chứa bằng chứng golden không
- 5 câu ngoài phạm vi có bị từ chối không; 20 câu trong phạm vi có bị từ chối nhầm không

So sánh 2 prompt: `basic` (quy tắc trích dẫn + từ chối) và `naive` (chỉ "trả lời dựa trên tài liệu").

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 100); pd.set_option("display.width", 220)

from ragbv.config import load_manifest, REPORTS, DATA
from ragbv import chunking as C, store, retrieval as R, generation as G
from ragbv.golden import load_golden, is_relevant

docs = load_manifest()
chunks = C.load_chunks("structure")
conn = store.connect()
ret = R.Retriever(chunks, docs, conn, "structure")
golden = load_golden()
print(len(golden), "câu:", sum(g.in_scope for g in golden), "trong phạm vi +", sum(not g.in_scope for g in golden), "ngoài phạm vi")

25 câu: 20 trong phạm vi + 5 ngoài phạm vi


## 1. Truy xuất + sinh

In [2]:
RETRIEVE = dict(method="hybrid3+rerank", k=5, n_cand=20, use_filter=True, use_rewrite=True)
contexts = {g.id: ret.run(g.question, **RETRIEVE) for g in golden}
runs = {}
for prompt in ("basic", "naive"):
    for g in golden:
        out = G.generate(g.question, contexts[g.id], prompt)
        runs[(prompt, g.id)] = {**out, **G.check(out, contexts[g.id])}
print("xong", len(runs))

C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   4%|▍         | 16/393 [00:00<00:02, 128.19it/s]

Loading weights:   7%|▋         | 29/393 [00:00<00:04, 74.14it/s] 

Loading weights:  10%|▉         | 38/393 [00:00<00:05, 68.47it/s]

Loading weights:  12%|█▏        | 46/393 [00:00<00:05, 65.55it/s]

... (6 outputs left out)

Loading weights:  26%|██▌       | 103/393 [00:01<00:05, 53.98it/s]

Loading weights:  28%|██▊       | 109/393 [00:01<00:05, 53.48it/s]

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 219.36it/s]

xong 50


In [3]:
def row(prompt, g):
    r, ctx = runs[(prompt, g.id)], contexts[g.id]
    pick = lambda ids: [ctx[int(c[1:]) - 1] for c in ids if c not in r["invalid_ids"]]
    gold_in_ctx = any(is_relevant(c.text, c.doc_id, e) for c in ctx for e in g.evidence) if g.in_scope else None
    hits = lambda cs: any(is_relevant(c.text, c.doc_id, e) for c in cs for e in g.evidence) if g.in_scope else None
    return {"prompt": prompt, "id": g.id, "type": g.type, "in_scope": g.in_scope, "status": r["status"],
            "refused": r["status"] == "khong_du_thong_tin", "has_citation": r["has_citation"],
            "invalid_ids": len(r["invalid_ids"]), "quote_verbatim": r["quote_verbatim_rate"],
            "quote_lenient": r["quote_lenient_rate"], "repaired": len(r["repaired"]),
            "gold_in_context": gold_in_ctx, "cites_gold": hits(pick(r["cited_ids"])),
            "cites_gold_repaired": hits(pick(r["cited_ids_repaired"])), "n_cited": len(r["cited_ids"]),
            "prompt_tokens": r["usage"]["prompt_tokens"], "completion_tokens": r["usage"]["completion_tokens"],
            "latency_s": r["latency_s"]}
table = pd.DataFrame([row(p, g) for p in ("basic", "naive") for g in golden])

def summarize(df):
    ins, oos = df[df.in_scope], df[~df.in_scope]
    answered = ins[~ins.refused]
    return pd.Series({
        "OOS từ chối đúng": f"{oos.refused.sum()}/{len(oos)}",
        "Trong phạm vi bị từ chối nhầm": f"{ins.refused.sum()}/{len(ins)}",
        "  trong đó ngữ cảnh có bằng chứng": int((ins.refused & ins.gold_in_context).sum()),
        "Câu trả lời có trích dẫn": f"{answered.has_citation.sum()}/{len(answered)}",
        "Trích dẫn id không tồn tại": int(df.invalid_ids.sum()),
        "Quote nguyên văn - chặt (TB)": round(df.quote_verbatim.dropna().mean(), 3),
        "Quote khớp từng phần (TB)": round(df.quote_lenient.dropna().mean(), 3),
        "Trích dẫn sai id, sửa được bằng quote": int(df.repaired.sum()),
        "Trích đúng đoạn bằng chứng": f"{answered.cites_gold.sum()}/{len(answered)}",
        "  sau khi sửa id": f"{answered.cites_gold_repaired.sum()}/{len(answered)}",
        "Ngữ cảnh có bằng chứng (retrieval)": f"{ins.gold_in_context.sum()}/{len(ins)}",
        "Token prompt TB": int(df.prompt_tokens.mean()), "Độ trễ LLM p50 (s)": round(df.latency_s.median(), 2)})
summary = pd.DataFrame({p: summarize(table[table.prompt == p]) for p in ("basic", "naive")})
summary

,basic,naive
OOS từ chối đúng,5/5,5/5
Trong phạm vi bị từ chối nhầm,0/20,0/20
trong đó ngữ cảnh có bằng chứng,0,0
Câu trả lời có trích dẫn,20/20,19/20
Trích dẫn id không tồn tại,0,0
Quote nguyên văn - chặt (TB),0.867,0.825
Quote khớp từng phần (TB),0.95,0.875
"Trích dẫn sai id, sửa được bằng quote",1,1
Trích đúng đoạn bằng chứng,18/20,18/20
sau khi sửa id,19/20,19/20


## 2. Từng câu

In [4]:
table[table.prompt == "basic"].drop(columns=["prompt", "prompt_tokens", "completion_tokens"])

,id,type,in_scope,status,refused,has_citation,invalid_ids,quote_verbatim,quote_lenient,repaired,gold_in_context,cites_gold,cites_gold_repaired,n_cited,latency_s
0,G01,table_lookup,True,tra_loi,False,True,0,0.000000,0.0,1,True,False,True,1,4.91
1,G02,table_lookup,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,1,4.37
2,G03,table_lookup,True,tra_loi,False,True,0,1.000000,1.0,0,False,False,False,3,5.43
3,G04,multi_condition,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,2,5.19
4,G05,fact,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,4,6.35
5,G06,fact,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,2,4.94
6,G07,fact,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,2,5.98
7,G08,table_lookup,True,tra_loi,False,True,0,0.000000,1.0,0,True,True,True,1,4.51
8,G09,multi_condition,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,3,6.89
9,G10,fact,True,tra_loi,False,True,0,1.000000,1.0,0,True,True,True,1,5.12


In [5]:
# Câu ngoài phạm vi: basic so với naive
for g in [x for x in golden if not x.in_scope]:
    for p in ("basic", "naive"):
        r = runs[(p, g.id)]
        print(f"[{g.id} | {p} | {r['status']}] {r['answer'][:260]}")
    print()

[O01 | basic | khong_du_thong_tin] Chưa có thông tin về phí bảo hiểm bắt buộc trách nhiệm dân sự xe ô tô của Bảo Việt năm 2026 trong các đoạn tài liệu đã cung cấp. Quý khách vui lòng liên hệ tư vấn viên để được hỗ trợ.
[O01 | naive | khong_du_thong_tin] Tài liệu cung cấp không có thông tin về phí bảo hiểm bắt buộc trách nhiệm dân sự xe ô tô của Bảo Việt năm 2026.

[O02 | basic | khong_du_thong_tin] Chưa có thông tin về sản phẩm An Phúc Gia Lộc và quyền lợi đáo hạn trong các đoạn tài liệu được cung cấp. Quý khách vui lòng liên hệ tư vấn viên để được hỗ trợ.
[O02 | naive | khong_du_thong_tin] Tài liệu cung cấp không đề cập đến quyền lợi đáo hạn của sản phẩm An Phúc Gia Lộc của Bảo Việt Nhân thọ.

[O03 | basic | khong_du_thong_tin] Chưa có thông tin về lãi suất tiền gửi kỳ hạn 12 tháng của Ngân hàng Bảo Việt trong các đoạn tài liệu đã cung cấp. Quý khách vui lòng liên hệ tư vấn viên để được hỗ trợ.
[O03 | naive | khong_du_thong_tin] Tài liệu cung cấp không đề cập đến lãi suất tiền gửi kỳ 

## 3. Báo cáo câu trả lời để duyệt tay + lưu

In [6]:
def answers_md(prompt):
    L = [f"# Lab 4 – Câu trả lời (prompt `{prompt}`)", "",
         "Sinh tự động. Mỗi câu: câu hỏi, đáp án chuẩn, câu trả lời của hệ thống, trích dẫn và kết quả hậu kiểm.", ""]
    for g in golden:
        r, ctx = runs[(prompt, g.id)], contexts[g.id]
        t = table[(table.prompt == prompt) & (table.id == g.id)].iloc[0]
        L += [f"## {g.id} · {g.type} · `{r['status']}`", "", f"**Câu hỏi:** {g.question}", "",
              f"**Đáp án chuẩn:** {g.answer}", "", f"**Hệ thống:** {r['answer']}", ""]
        flags = [f"ngữ cảnh có bằng chứng: {t.gold_in_context}", f"trích đúng đoạn bằng chứng: {t.cites_gold}",
                 f"quote chặt: {t.quote_verbatim}", f"quote từng phần: {t.quote_lenient}",
                 f"id sửa lại: {r['repaired'] or 'không'}", f"id không tồn tại: {r['invalid_ids'] or 'không'}"]
        L += ["*Hậu kiểm:* " + " · ".join(flags), "", "| Trích dẫn | Tài liệu · trang | Quote |", "|---|---|---|"]
        for c in r["citations"]:
            ok = c["id"] in {f"C{i}" for i in range(1, len(ctx) + 1)}
            ch = ctx[int(c["id"][1:]) - 1] if ok else None
            where = f"`{ch.doc_id}` · tr.{','.join(ch.page_labels)}" if ch else "**không tồn tại**"
            L.append(f"| {c['id']} | {where} | {c['quote'].replace('|', '/')} |")
        L.append("")
    return "\n".join(L) + "\n"

for p in ("basic", "naive"):
    (REPORTS / f"lab4_answers_{p}.md").write_text(answers_md(p), encoding="utf-8")

runs_dir = DATA / "runs"; runs_dir.mkdir(exist_ok=True)
with open(runs_dir / "lab4_basic.jsonl", "w", encoding="utf-8") as f:
    for g in golden:
        r = runs[("basic", g.id)]
        f.write(json.dumps({"id": g.id, "type": g.type, "question": g.question, "reference": g.answer,
                            "answer": r["answer"], "status": r["status"],
                            "contexts": [c.text for c in contexts[g.id]],
                            "context_ids": [c.chunk_id for c in contexts[g.id]],
                            "context_docs": [c.doc_id for c in contexts[g.id]]}, ensure_ascii=False) + "\n")
out = {"retrieve": RETRIEVE, "summary": summary.to_dict(), "per_question": table.to_dict("records")}
(REPORTS / "lab4_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
